# EdgeCoRAG 已保存记录诊断

## tl;dr

- 正式主条件17745条回答对齐为5915个问题×模型单元，不能当成5915道独立问题。
- Qwen3-VL 4B的S1−S0净变化为−26题；无GT哈希命中组−30题，命中组+4题。
- Gemma E2B/E4B在S2仍错592/540题，且格式失败为0；不能把主要差距归因于格式。
- Qwen3.5的S2全部55次格式错误即使假想补对，最高也只有656/1033=63.50%。这不是恢复评分或新预算实验预测。
- 本笔记本只核验记录，不作看图机制审定。

## Context & Methods

本轮在固定运行之后做探索性分解，S0/S1/S2分别为无证据、发布候选top3、actual GT。内部留出150题与冻结1033题分开。

### Key Assumptions

- 以split/model/qid为三条件配对单元，严格解析器和全题单分母不变。
- GT哈希命中只比较已提供JPEG95图片的SHA，不等于语义支持或检索正确。
- 格式状态分组、GT分组分别可加；两种分解不是互斥的因果类别。
- 不合并S3、性能、smoke、development或warmup；没有新增推理、下载、参数调整或新的显著性检验。
- 未核验正式原图；新病例为事后固定规则选取，视觉审定字段始终false。

### 1. 载入可复算的诊断模块

In [1]:
import json, hashlib
from pathlib import Path
import pandas as pd
import diagnose_saved_runs as diag
result = diag.compute()
result['audit']

{'status': 'PASS_records_only',
 'quality_main_question_runs': 17745,
 'paired_question_model_units': 5915,
 'frozen_unique_questions': 1033,
 'holdout_unique_questions': 150,
 'models': 5,
 'condition_groups': 30,
 'technical_failures_in_main_quality': 0,
 'main_quality_format_failures': 99,
 'format_failures_at_token_limit': 98,
 'sql_reconciled': ['patterns',
  'format_decomposition',
  'gt_decomposition',
  'condition_errors',
  'record_summary',
  'selected_cases'],
 'original_quality_and_transition_tables_reconciled': True,
 'selected_case_cards': 20,
 'visual_mechanism_reviews_completed': 0,
 'new_inference': False,
 'network_access': False,
 'scientific_parameters_changed': False,
 'post_hoc_analysis': True,
 'causal_claims_supported': False,
 'quality_s3_excluded_from_this_diagnostic': 62,
 'performance_smoke_dev_preflight_excluded': True}

## Data

输入是仓库内logs/B_predictions.jsonl.gz与B_input_contexts.jsonl.gz。来源为runtime-v1.10.1的已核验归档。以下保存输入哈希，不依赖本机绝对目录。

In [2]:
source_hashes = {str(p.relative_to(diag.REPO)): hashlib.sha256(p.read_bytes()).hexdigest()
                 for p in [diag.REPO/'logs/B_predictions.jsonl.gz', diag.REPO/'logs/B_input_contexts.jsonl.gz']}
assert result['audit']['paired_question_model_units'] == 5915
assert result['audit']['quality_main_question_runs'] == 17745
assert result['audit']['visual_mechanism_reviews_completed'] == 0
source_hashes

{'logs/B_predictions.jsonl.gz': 'df22ac5f70ce127629dbc807fd389af32282613c3d95815a68fe20760bf63c38',
 'logs/B_input_contexts.jsonl.gz': 'f69c1f0a77a28e56470b1f53a2bb1cde900e36b7940a6ea3053fddb78255215f'}

## Results

### 2. 核对GT分组对总体变化的贡献

净答对数除以完整1033题，所以两组贡献相加可恢复总体S1−S0，不是成功子集准确率。

In [3]:
gt = pd.DataFrame(result['gt_decomposition'])
frozen_gt = gt[gt.split.eq('frozen_test')]
frozen_gt[['model','group','n','improvements','regressions','net_correct_change','contribution_pp_full_denominator']]

,model,group,n,improvements,regressions,net_correct_change,contribution_pp_full_denominator
0,Gemma4 E2B,无GT哈希命中,517,46,60,-14,-1.355276
1,Gemma4 E2B,至少一个GT哈希命中,516,57,52,5,0.484027
2,Gemma4 E4B,无GT哈希命中,517,55,49,6,0.580833
3,Gemma4 E4B,至少一个GT哈希命中,516,61,37,24,2.323330
4,Qwen3.5 4B,无GT哈希命中,517,49,63,-14,-1.355276
5,Qwen3.5 4B,至少一个GT哈希命中,516,61,37,24,2.323330
6,Qwen3-VL 4B,无GT哈希命中,517,37,67,-30,-2.904163
7,Qwen3-VL 4B,至少一个GT哈希命中,516,42,38,4,0.387222
8,Qwen3-VL 8B,无GT哈希命中,517,34,57,-23,-2.226525
9,Qwen3-VL 8B,至少一个GT哈希命中,516,42,22,20,1.936108


### 3. 查看三条件模式与GT条件残余错误

In [4]:
summary = pd.DataFrame(result['summary'])
summary[summary.split.eq('frozen_test')][['model','all_three_wrong','s1_wrong_s2_correct','s1_wrong_s2_correct_with_gt_hash_hit','s2_wrong']]

,model,all_three_wrong,s1_wrong_s2_correct,s1_wrong_s2_correct_with_gt_hash_hit,s2_wrong
0,Gemma4 E2B,469,141,58,592
1,Gemma4 E4B,437,128,53,540
2,Qwen3.5 4B,286,124,37,432
3,Qwen3-VL 4B,238,161,47,316
4,Qwen3-VL 8B,277,143,44,343


### 4. 格式分解与极端上限

极端上限仅固定其他回答，并把格式失败全部假想算对；不重新解析原文，也不是增加token的效果估计。

In [5]:
bridge = pd.DataFrame(result['format_decomposition'])
bridge[(bridge.split=='frozen_test') & (bridge.model_key=='qwen35_4b') & (bridge.left=='S1') & (bridge.right=='S2')][['format_group','n','improvements','regressions','net_correct_change','contribution_pp_full_denominator']]

,format_group,n,improvements,regressions,net_correct_change,contribution_pp_full_denominator
28,both_parse_ok,964,117,61,56,5.421104
29,right_only_format_failure,48,0,25,-25,-2.420136
30,left_only_format_failure,14,7,0,7,0.677638
31,both_format_failure,7,0,0,0,0.000000


In [6]:
errors = pd.DataFrame(result['condition_errors'])
errors[(errors.split=='frozen_test') & (errors.condition=='S2')][['model','n','correct','valid_choice_wrong','format_failures','strict_accuracy','all_format_failures_assumed_correct_ceiling']]

,model,n,correct,valid_choice_wrong,format_failures,strict_accuracy,all_format_failures_assumed_correct_ceiling
2,Gemma4 E2B,1033,441,592,0,0.426912,0.426912
5,Gemma4 E4B,1033,493,540,0,0.477251,0.477251
8,Qwen3.5 4B,1033,601,377,55,0.581801,0.635044
11,Qwen3-VL 4B,1033,717,316,0,0.694095,0.694095
14,Qwen3-VL 8B,1033,690,343,0,0.667957,0.667957


### 5. 查看待审定的记录案例

每模型每类按SHA256固定顺序选一题，避免同模型重复qid。选例在看到运行结果之后完成；不是随机代表样本，没有图像语义审定。下表仅示范一类，完整20例在配套CSV。

In [7]:
cases = pd.DataFrame(result['selected_cases'])
cases[cases.category.eq('hash_hit_s1_wrong_s2_correct')][['model','qid','question','gold','S0_raw','S1_raw','S2_raw','visual_review_completed']]

,model,qid,question,gold,S0_raw,S1_raw,S2_raw,visual_review_completed
1,Qwen3-VL 4B,mragb:396,Can you identify what type of animal this is?,C,C,B,C,False
5,Qwen3-VL 8B,mragb:1329,From which region is this object sourced?,C,B,B,C,False
9,Gemma4 E2B,mragb:232,Which animal is this?,A,B,B,A,False
13,Gemma4 E4B,mragb:674,"Among the following characteristics, which is ...",D,B,A,D,False
17,Qwen3.5 4B,mragb:562,Which particular model of car is this?,C,C\n,B\n,C\n,False


### 6. 验证SQL对账与摘要数字

In [8]:
assert len(result['audit']['sql_reconciled']) == 6
assert all(not r['mechanism_adjudicated'] and not r['visual_review_completed'] for r in result['case_ledger'])
q4 = frozen_gt[frozen_gt.model_key.eq('qwen3vl_4b')]
assert sorted(q4.net_correct_change.tolist()) == [-30, 4]
assert q4.net_correct_change.sum() == -26
q35 = errors[(errors.split=='frozen_test') & (errors.model_key=='qwen35_4b') & (errors.condition=='S2')].iloc[0]
assert q35.correct + q35.format_failures == 656
assert q35.all_format_failures_assumed_correct_ceiling < 717/1033
{'status':'PASS', 'SQL_groups_reconciled':result['audit']['sql_reconciled'], 'visual_reviews':0}

{'status': 'PASS',
 'SQL_groups_reconciled': ['patterns',
  'format_decomposition',
  'gt_decomposition',
  'condition_errors',
  'record_summary',
  'selected_cases'],
 'visual_reviews': 0}

## Takeaways

记录能确定退步出现在哪些分组、格式失败最多影响多少固定记录分数、哪些题适合后续审查；不能单凭这些记录确定检索失败、证据利用失败或纯推理不足。

S1与S2的图片数量不同，GT集合命中不一定完整，题目难度和场景也可能混杂。来源依赖与审计暴露保持披露，不新增来源聚类或因果结论。下一步若要判定机制，需要固定原图和单独的审阅规范；这不授权重跑或调本轮prompt/预算。